# AAAI 2026 Nesy-Gen Revision Experiments

This Colab notebook runs the switchable paper-grade pipeline for IU-Xray official split and MIMIC-CXR:

- Vision-T5 image-to-report generator
- RAG/retrieval report candidates
- PrimeKG entity grounding
- optional soft graph-constrained decoding
- LTN-style ante-hoc graph reasoning
- Consistency Gate candidate selection
- entity-linking validation
- official metric input preparation

Set `RUN_DATASET` and `DECODING_MODE` in Cell 2.

## Cell 1: Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## Cell 2: Main Configuration

In [ ]:
from pathlib import Path

RUN_DATASET = "iuxray_official"  # options: "iuxray_official", "mimic_aug"
DECODING_MODE = "graph_constrained"  # options: "standard", "graph_constrained"

RUN_NAME = "aaai_vision_t5_base_convnext_primekg"
SEED = 13
SMOKE_RUN = False

REPO_DIR = Path("/content/Nesy-GenRevision")
PRIMEKG_SOURCE_DIR = Path("/content/drive/MyDrive/dataverse_files")

AAAI_ROOT = Path("/content/drive/MyDrive/aaai_2026_experiments")
OUTPUT_DIR = AAAI_ROOT / RUN_DATASET / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAD_PRIMEKG_DIR = Path(f"/content/drive/MyDrive/primekg_radiology_cache_{RUN_DATASET}")

# Full AAAI run defaults. For a quick test, set SMOKE_RUN=True.
MAX_TRAIN_EXAMPLES = 256 if SMOKE_RUN else None
MAX_VAL_EXAMPLES = 64 if SMOKE_RUN else None
MAX_TEST_EXAMPLES = 100 if SMOKE_RUN else None

# --- Vision-T5 generator (A100 profile; BLEU-1 > 0.5 is a validation-led target) ---
TEXT_MODEL_NAME = "google/flan-t5-base"
TOKENIZER_NAME = TEXT_MODEL_NAME
VISION_BACKBONE = "convnext_base"

# Keep the pretrained vision encoder frozen for the main paper setting.
# Only the visual projection and text decoder are trained.
# Set False only for an explicit unfreeze ablation.
FREEZE_VISUAL_ENCODER = True
LEARNING_RATE = 5e-5           # T5 + visual projection
BACKBONE_LEARNING_RATE = 5e-6  # used only if FREEZE_VISUAL_ENCODER=False

IMAGE_SIZE = 224
MAX_TARGET_LENGTH = 192
VISUAL_SEQ_LEN = 128           # kept for config compat; not used by spatial encoder

# --- Speed settings (A100) ----------------------------------------------------
# BF16 is native on A100 — faster and more stable than FP16 (no GradScaler).
# Batch 16 + accum 1 = same effective batch as 4+accum 4, but 4x fewer steps.
# num_workers=4 loads images in parallel while GPU runs.
# torch.compile graph-optimises T5 (~20% faster forward/backward).
TRAIN_EPOCHS = 10 if not SMOKE_RUN else 1  # best validation epoch is retained
TRAIN_BATCH_SIZE = 16 if not SMOKE_RUN else 4
GRADIENT_ACCUMULATION_STEPS = 1
NUM_WORKERS = 4
USE_BF16 = True          # set False if you hit a bf16 compatibility error
COMPILE_MODEL = True     # requires PyTorch >= 2.0 (Colab A100 always has it)

# Candidate and retrieval settings.
RETRIEVAL_TOP_K = 10
RETRIEVAL_MODE = "visual"  # frozen-image retrieval; never queries with the test report
RETRIEVAL_CONDITIONING = True
RETRIEVAL_EVIDENCE_TOP_K = 3
MAX_EVIDENCE_LENGTH = 192
GENERATOR_NUM_CANDIDATES = 8
GENERATOR_NUM_BEAMS = 8
GENERATOR_BATCH_SIZE = 2
MAX_NEW_TOKENS = 140

# Diverse decoding / anti-template controls.
GENERATOR_REPETITION_PENALTY = 1.12
GENERATOR_NO_REPEAT_NGRAM_SIZE = 3
GENERATOR_LENGTH_PENALTY = 0.90
GENERATOR_NUM_BEAM_GROUPS = 1  # compatible with custom PrimeKG logits processing
GENERATOR_DIVERSITY_PENALTY = 0.0

# Soft graph-constrained decoding.
GRAPH_TOKEN_BOOST = 1.5
UNSUPPORTED_TOKEN_PENALTY = 0.0
CONSTRAINT_MAX_TERMS = 2500

# Let Vision-T5 drive candidate selection (higher graph weight than retrieval).
SELECTION_OBJECTIVE = "hybrid" if DECODING_MODE == "graph_constrained" else "graph"
GRAPH_SCORE_WEIGHT = 0.55
EVIDENCE_WEIGHT = 0.35
GATE_WEIGHT = 0.10
GENERATED_EVIDENCE_SCORE = 0.55

# Graph-aware training.
GRAPH_TRAINING_MODE = "primekg_token"
GRAPH_TOKEN_LOSS_WEIGHT = 0.05
UNSUPPORTED_TOKEN_LOSS_WEIGHT = 0.00
GRAPH_LOSS_MAX_TERMS = 2500

GENERATOR_NAME = "rag_primekg_constrained" if DECODING_MODE == "graph_constrained" else "rag_primekg"

print("Dataset:", RUN_DATASET)
print("Decoding:", DECODING_MODE)
print("Backbone:", VISION_BACKBONE, "frozen:", FREEZE_VISUAL_ENCODER)
print("Text model:", TEXT_MODEL_NAME)
print(f"Batch: {TRAIN_BATCH_SIZE} x grad_accum {GRADIENT_ACCUMULATION_STEPS} "
      f"| workers: {NUM_WORKERS} | bf16: {USE_BF16} | compile: {COMPILE_MODEL}")
print("Epochs:", TRAIN_EPOCHS)
print("LR:", LEARNING_RATE, "/ backbone:", BACKBONE_LEARNING_RATE)
print("Selection:", SELECTION_OBJECTIVE,
      f"(graph={GRAPH_SCORE_WEIGHT}, evidence={EVIDENCE_WEIGHT})")
print("Output:", OUTPUT_DIR)


## Cell 3: Clone/Pull Repo And Install Dependencies

In [ ]:
import json
import subprocess
import sys

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/FaezehMillerAI/Nesy-GenRevision.git", str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[torch,eval,colab]"],
    check=True,
)

print("Repo ready:", REPO_DIR)

## Cell 4: GPU Check

In [ ]:
import torch

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Cell 5: Resolve Dataset Root

In [ ]:
from pathlib import Path

if RUN_DATASET == "iuxray_official":
    DATA_ROOT = Path("/content/drive/MyDrive/iuxray")
    MANIFEST = OUTPUT_DIR / "iuxray_official_manifest.jsonl"

elif RUN_DATASET == "mimic_aug":
    MIMIC_DRIVE_ROOT = Path("/content/drive/MyDrive/mimic_cxr_dataset/versions/2")
    if MIMIC_DRIVE_ROOT.exists():
        DATA_ROOT = MIMIC_DRIVE_ROOT
    else:
        import kagglehub
        DATA_ROOT = Path(kagglehub.dataset_download("simhadrisadaram/mimic-cxr-dataset"))
        while not (DATA_ROOT / "mimic_cxr_aug_train.csv").exists() and DATA_ROOT.parent != DATA_ROOT:
            DATA_ROOT = DATA_ROOT.parent
    MANIFEST = OUTPUT_DIR / "mimic_aug_manifest.jsonl"

else:
    raise ValueError(RUN_DATASET)

print("DATA_ROOT:", DATA_ROOT)
print("Exists:", DATA_ROOT.exists())
print("MANIFEST:", MANIFEST)

## Cell 6: Build Dataset Manifest

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable, "scripts/build_manifest.py",
        "--dataset", RUN_DATASET,
        "--data-root", str(DATA_ROOT),
        "--output", str(MANIFEST),
        "--seed", str(SEED),
    ],
    cwd=REPO_DIR,
    check=True,
)

print("Manifest exists:", MANIFEST.exists())

## Cell 7: Inspect Manifest

In [ ]:
import json
from collections import Counter

rows = [json.loads(line) for line in MANIFEST.read_text().splitlines()]
print("Examples:", len(rows))
print("Splits:", Counter(row["split"] for row in rows))
rows[0]

## Cell 8: Build Or Reuse Radiology PrimeKG Cache

In [ ]:
import json
import subprocess
import sys

CACHE_SUMMARY = RAD_PRIMEKG_DIR / "radiology_primekg_summary.json"
cache_is_train_only = False
if CACHE_SUMMARY.exists():
    cache_is_train_only = json.loads(CACHE_SUMMARY.read_text()).get("seed_split") == "train"
if (not cache_is_train_only or not (RAD_PRIMEKG_DIR / "kg.csv").exists()
        or not (RAD_PRIMEKG_DIR / "nodes.csv").exists()):
    subprocess.run(
        [
            sys.executable, "scripts/build_radiology_primekg.py",
            "--primekg-dir", str(PRIMEKG_SOURCE_DIR),
            "--manifest", str(MANIFEST),
            "--output-dir", str(RAD_PRIMEKG_DIR),
            "--hops", "1",
            "--seed-split", "train",
        ],
        cwd=REPO_DIR,
        check=True,
    )

print("PrimeKG cache:", RAD_PRIMEKG_DIR)
print((RAD_PRIMEKG_DIR / "radiology_primekg_summary.json").read_text())

## Cell 9: Entity Extraction/Linking Validation

In [ ]:
import subprocess
import sys

ENTITY_VALIDATION_DIR = OUTPUT_DIR / "entity_linking_validation"

subprocess.run(
    [
        sys.executable, "scripts/validate_entity_linking.py",
        "--manifest", str(MANIFEST),
        "--nodes-csv", str(RAD_PRIMEKG_DIR / "nodes.csv"),
        "--output-dir", str(ENTITY_VALIDATION_DIR),
        "--prefix", RUN_DATASET,
        "--split", "test",
        "--limit", str(MAX_TEST_EXAMPLES or 1000),
        "--audit-sample-size", "100",
        "--seed", str(SEED),
    ],
    cwd=REPO_DIR,
    check=True,
)

print("Entity validation:", ENTITY_VALIDATION_DIR)

## Cell 10: Train Vision-T5 Generator

In [ ]:
import os
import subprocess
import sys

VISION_T5_CKPT = OUTPUT_DIR / "checkpoints" / f"vision_t5_{VISION_BACKBONE}_{TEXT_MODEL_NAME.split('/')[-1]}"
TRAIN_LOG = OUTPUT_DIR / f"{RUN_DATASET}_{GENERATOR_NAME}_training.log"

cmd = [
    sys.executable, "-u", "scripts/train_vision_t5_generator.py",
    "--manifest", str(MANIFEST),
    "--output-dir", str(VISION_T5_CKPT),
    "--text-model-name", TEXT_MODEL_NAME,
    "--tokenizer-name", TOKENIZER_NAME,
    "--visual-backbone", VISION_BACKBONE,
    "--image-size", str(IMAGE_SIZE),
    "--epochs", str(TRAIN_EPOCHS),
    "--batch-size", str(TRAIN_BATCH_SIZE),
    "--gradient-accumulation-steps", str(GRADIENT_ACCUMULATION_STEPS),
    "--learning-rate", str(LEARNING_RATE),
    "--max-target-length", str(MAX_TARGET_LENGTH),
    "--visual-seq-len", str(VISUAL_SEQ_LEN),
    "--num-workers", str(NUM_WORKERS),
    "--progress-style", "tqdm",
    "--progress-every", "1",
    "--seed", str(SEED),
]

if RETRIEVAL_CONDITIONING:
    cmd += [
        "--retrieval-conditioning",
        "--retrieval-top-k", str(RETRIEVAL_EVIDENCE_TOP_K),
        "--max-evidence-length", str(MAX_EVIDENCE_LENGTH),
    ]

if USE_BF16:
    cmd += ["--bf16"]
else:
    cmd += ["--fp16"]

if COMPILE_MODEL:
    cmd += ["--compile-model"]

if FREEZE_VISUAL_ENCODER:
    cmd += ["--freeze-visual-encoder"]
elif BACKBONE_LEARNING_RATE is not None:
    cmd += ["--backbone-learning-rate", str(BACKBONE_LEARNING_RATE)]

if MAX_TRAIN_EXAMPLES:
    cmd += ["--max-train-examples", str(MAX_TRAIN_EXAMPLES)]
if MAX_VAL_EXAMPLES:
    cmd += ["--max-val-examples", str(MAX_VAL_EXAMPLES)]

if GRAPH_TRAINING_MODE != "none":
    cmd += [
        "--graph-training-mode", GRAPH_TRAINING_MODE,
        "--graph-loss-nodes-csv", str(RAD_PRIMEKG_DIR / "nodes.csv"),
        "--graph-token-loss-weight", str(GRAPH_TOKEN_LOSS_WEIGHT),
        "--unsupported-token-loss-weight", str(UNSUPPORTED_TOKEN_LOSS_WEIGHT),
        "--graph-loss-max-terms", str(GRAPH_LOSS_MAX_TERMS),
        "--graph-loss-source", "indication_reference",
    ]

print("Running training command:")
print(" ".join(cmd))
print("Log:", TRAIN_LOG)
print("-" * 100)

TRAIN_LOG.parent.mkdir(parents=True, exist_ok=True)
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["PYTHONIOENCODING"] = "utf-8"

with TRAIN_LOG.open("w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        cmd,
        cwd=REPO_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=0,
        env=env,
    )
    while True:
        chunk = process.stdout.read(1)
        if chunk == "" and process.poll() is not None:
            break
        if chunk:
            print(chunk, end="", flush=True)
            log_file.write(chunk)
            log_file.flush()
    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(f"Training failed with exit code {return_code}. See log: {TRAIN_LOG}")

print("\nCheckpoint:", VISION_T5_CKPT)


## Cell 11: Run RAG + PrimeKG + LTN + Consistency Gate

In [ ]:
import os
import subprocess
import sys

RAG_OUTPUT_CSV = OUTPUT_DIR / f"{RUN_DATASET}_{GENERATOR_NAME}_predictions.csv"
RAG_CANDIDATES_CSV = OUTPUT_DIR / f"{RUN_DATASET}_{GENERATOR_NAME}_candidate_audit.csv"
GENERATION_LOG = OUTPUT_DIR / f"{RUN_DATASET}_{GENERATOR_NAME}_generation.log"

cmd = [
    sys.executable, "-u", "scripts/generate_rag_primekg_reports.py",
    "--manifest", str(MANIFEST),
    "--primekg-dir", str(RAD_PRIMEKG_DIR),
    "--output-csv", str(RAG_OUTPUT_CSV),
    "--candidates-csv", str(RAG_CANDIDATES_CSV),
    "--split", "test",
    "--retrieval-top-k", str(RETRIEVAL_TOP_K),
    "--retrieval-mode", RETRIEVAL_MODE,
    "--generator-checkpoint-dir", str(VISION_T5_CKPT),
    "--generator-num-candidates", str(GENERATOR_NUM_CANDIDATES),
    "--generated-evidence-score", str(GENERATED_EVIDENCE_SCORE),
    "--generator-num-beams", str(GENERATOR_NUM_BEAMS),
    "--generator-batch-size", str(GENERATOR_BATCH_SIZE),
    "--generator-repetition-penalty", str(GENERATOR_REPETITION_PENALTY),
    "--generator-no-repeat-ngram-size", str(GENERATOR_NO_REPEAT_NGRAM_SIZE),
    "--generator-length-penalty", str(GENERATOR_LENGTH_PENALTY),
    "--generator-num-beam-groups", str(GENERATOR_NUM_BEAM_GROUPS),
    "--generator-diversity-penalty", str(GENERATOR_DIVERSITY_PENALTY),
    "--max-new-tokens", str(MAX_NEW_TOKENS),
    "--decoding-mode", DECODING_MODE,
    "--graph-token-boost", str(GRAPH_TOKEN_BOOST),
    "--unsupported-token-penalty", str(UNSUPPORTED_TOKEN_PENALTY),
    "--constraint-max-terms", str(CONSTRAINT_MAX_TERMS),
    "--subgraph-strategy", "ego",
    "--min-graph-score", "0.50",
    "--selection-objective", SELECTION_OBJECTIVE,
    "--graph-score-weight", str(GRAPH_SCORE_WEIGHT),
    "--evidence-weight", str(EVIDENCE_WEIGHT),
    "--gate-weight", str(GATE_WEIGHT),
]

if MAX_TEST_EXAMPLES:
    cmd += ["--limit", str(MAX_TEST_EXAMPLES)]

print("Streaming generation progress. This runs retrieval, candidate generation, then graph verification.")
print("Log:", GENERATION_LOG)
print("-" * 100)

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["PYTHONIOENCODING"] = "utf-8"
GENERATION_LOG.parent.mkdir(parents=True, exist_ok=True)

with GENERATION_LOG.open("w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        cmd,
        cwd=REPO_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=0,
        env=env,
    )
    while True:
        chunk = process.stdout.read(1)
        if chunk == "" and process.poll() is not None:
            break
        if chunk:
            print(chunk, end="", flush=True)
            log_file.write(chunk)
            log_file.flush()
    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(f"Generation failed with exit code {return_code}. See log: {GENERATION_LOG}")

print("Predictions:", RAG_OUTPUT_CSV)
print("Candidate audit:", RAG_CANDIDATES_CSV)


## Cell 12: Quick Internal Evaluation

This gives fast development feedback. For final paper numbers, use the official outputs prepared in Cell 13 and evaluated in Cell 14.

In [ ]:
import subprocess
import sys

EVAL_JSON = OUTPUT_DIR / f"{RUN_DATASET}_{GENERATOR_NAME}_quick_eval.json"

subprocess.run(
    [
        sys.executable, "scripts/evaluate_generation.py",
        "--manifest", str(MANIFEST),
        "--predictions-csv", str(RAG_OUTPUT_CSV),
        "--output-json", str(EVAL_JSON),
        "--nodes-csv", str(RAD_PRIMEKG_DIR / "nodes.csv"),
        "--output-factuality-csv", str(OUTPUT_DIR / f"{RUN_DATASET}_{GENERATOR_NAME}_entity_factuality.csv"),
        "--output-chexpert-csv", str(OUTPUT_DIR / f"{RUN_DATASET}_{GENERATOR_NAME}_chexpert_lite.csv"),
        "--output-radgraph-csv", str(OUTPUT_DIR / f"{RUN_DATASET}_{GENERATOR_NAME}_radgraph_lite.csv"),
    ],
    cwd=REPO_DIR,
    check=True,
)

print(EVAL_JSON.read_text())

## Cell 13: Leakage / Reference-Overlap Audit

Run this before trusting unusually high BLEU/ROUGE scores.


In [ ]:
import subprocess
import sys

LEAKAGE_AUDIT_JSON = OUTPUT_DIR / f"{RUN_DATASET}_{GENERATOR_NAME}_leakage_audit.json"

subprocess.run(
    [
        sys.executable, "scripts/audit_prediction_leakage.py",
        "--manifest", str(MANIFEST),
        "--predictions-csv", str(RAG_OUTPUT_CSV),
        "--output-json", str(LEAKAGE_AUDIT_JSON),
        "--high-overlap-threshold", "0.95",
    ],
    cwd=REPO_DIR,
    check=True,
)

print(LEAKAGE_AUDIT_JSON.read_text())

## Cell 14: Prepare Official Metric Inputs

In [ ]:
import subprocess
import sys

OFFICIAL_INPUT_DIR = OUTPUT_DIR / f"official_metric_inputs_{GENERATOR_NAME}"

subprocess.run(
    [
        sys.executable, "scripts/evaluate_generation.py",
        "--manifest", str(MANIFEST),
        "--predictions-csv", str(RAG_OUTPUT_CSV),
        "--output-json", str(OUTPUT_DIR / f"{RUN_DATASET}_{GENERATOR_NAME}_official_input_prep.json"),
        "--prepare-official-inputs-dir", str(OFFICIAL_INPUT_DIR),
        "--nodes-csv", str(RAD_PRIMEKG_DIR / "nodes.csv"),
    ],
    cwd=REPO_DIR,
    check=True,
)

print("Official inputs:", OFFICIAL_INPUT_DIR)
print("CheXbert pred:", OFFICIAL_INPUT_DIR / "pred_reports_for_chexbert.csv")
print("CheXbert ref:", OFFICIAL_INPUT_DIR / "ref_reports_for_chexbert.csv")
print("RadGraph input:", OFFICIAL_INPUT_DIR / "reports_for_radgraph.json")

## Cell 15: Official Evaluation After External CheXbert/RadGraph

Run the official CheXbert/CheXpert and RadGraph tools on the files from Cell 13, save their outputs into `OFFICIAL_OUTPUT_DIR`, then rerun this cell.

In [ ]:
import subprocess
import sys
from pathlib import Path

OFFICIAL_OUTPUT_DIR = OUTPUT_DIR / f"official_metric_outputs_{GENERATOR_NAME}"

CHEXBERT_PRED_LABELS_CSV = OFFICIAL_OUTPUT_DIR / "pred_chexbert_labels.csv"
CHEXBERT_REF_LABELS_CSV = OFFICIAL_OUTPUT_DIR / "ref_chexbert_labels.csv"
RADGRAPH_PRED_JSON = OFFICIAL_OUTPUT_DIR / "pred_radgraph.json"
RADGRAPH_REF_JSON = OFFICIAL_OUTPUT_DIR / "ref_radgraph.json"

required = [
    CHEXBERT_PRED_LABELS_CSV,
    CHEXBERT_REF_LABELS_CSV,
    RADGRAPH_PRED_JSON,
    RADGRAPH_REF_JSON,
]

if not all(path.exists() for path in required):
    print("Official external outputs are not ready yet.")
    for path in required:
        print(path, path.exists())
else:
    subprocess.run(
        [
            sys.executable, "scripts/evaluate_generation.py",
            "--manifest", str(MANIFEST),
            "--predictions-csv", str(RAG_OUTPUT_CSV),
            "--output-json", str(OUTPUT_DIR / f"{RUN_DATASET}_{GENERATOR_NAME}_official_metrics.json"),
            "--official-coco",
            "--official-only",
            "--chexbert-pred-labels-csv", str(CHEXBERT_PRED_LABELS_CSV),
            "--chexbert-ref-labels-csv", str(CHEXBERT_REF_LABELS_CSV),
            "--radgraph-pred-json", str(RADGRAPH_PRED_JSON),
            "--radgraph-ref-json", str(RADGRAPH_REF_JSON),
            "--output-official-chexbert-csv", str(OUTPUT_DIR / f"{RUN_DATASET}_{GENERATOR_NAME}_official_chexbert.csv"),
            "--output-official-radgraph-csv", str(OUTPUT_DIR / f"{RUN_DATASET}_{GENERATOR_NAME}_official_radgraph.csv"),
            "--nodes-csv", str(RAD_PRIMEKG_DIR / "nodes.csv"),
            "--output-factuality-csv", str(OUTPUT_DIR / f"{RUN_DATASET}_{GENERATOR_NAME}_official_entity_factuality.csv"),
        ],
        cwd=REPO_DIR,
        check=True,
    )

## Cell 16: Save Reproducible Experiment Config

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, "scripts/write_experiment_config.py",
    "--output", str(OUTPUT_DIR / f"{RUN_DATASET}_{GENERATOR_NAME}_experiment_config.json"),
    "--dataset", RUN_DATASET,
    "--manifest", str(MANIFEST),
    "--primekg-dir", str(RAD_PRIMEKG_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--run-name", RUN_NAME,
    "--generator", GENERATOR_NAME,
    "--generator-checkpoint-dir", str(VISION_T5_CKPT),
    "--retrieval-top-k", str(RETRIEVAL_TOP_K),
    "--retrieval-mode", RETRIEVAL_MODE,
    "--generator-num-candidates", str(GENERATOR_NUM_CANDIDATES),
    "--vision-backbone", VISION_BACKBONE,
    "--text-model-name", TEXT_MODEL_NAME,
    "--decoding-mode", DECODING_MODE,
    "--generator-repetition-penalty", str(GENERATOR_REPETITION_PENALTY),
    "--generator-no-repeat-ngram-size", str(GENERATOR_NO_REPEAT_NGRAM_SIZE),
    "--generator-length-penalty", str(GENERATOR_LENGTH_PENALTY),
    "--generator-num-beam-groups", str(GENERATOR_NUM_BEAM_GROUPS),
    "--generator-diversity-penalty", str(GENERATOR_DIVERSITY_PENALTY),
    "--graph-token-boost", str(GRAPH_TOKEN_BOOST),
    "--unsupported-token-penalty", str(UNSUPPORTED_TOKEN_PENALTY),
    "--selection-objective", SELECTION_OBJECTIVE,
    "--graph-score-weight", str(GRAPH_SCORE_WEIGHT),
    "--evidence-weight", str(EVIDENCE_WEIGHT),
    "--gate-weight", str(GATE_WEIGHT),
    "--graph-training-mode", GRAPH_TRAINING_MODE,
    "--graph-token-loss-weight", str(GRAPH_TOKEN_LOSS_WEIGHT),
    "--unsupported-token-loss-weight", str(UNSUPPORTED_TOKEN_LOSS_WEIGHT),
    "--subgraph-strategy", "ego",
    "--official-metrics",
]
if FREEZE_VISUAL_ENCODER:
    cmd += ["--freeze-visual-encoder"]
if RETRIEVAL_CONDITIONING:
    cmd += ["--retrieval-conditioning"]

subprocess.run(cmd, cwd=REPO_DIR, check=True)


## Cell 17: Inspect Outputs

In [ ]:
import pandas as pd

print("Predictions:", RAG_OUTPUT_CSV)
print("Candidates:", RAG_CANDIDATES_CSV)

pred_df = pd.read_csv(RAG_OUTPUT_CSV)
cand_df = pd.read_csv(RAG_CANDIDATES_CSV)

print("Prediction rows:", len(pred_df))
print("Candidate rows:", len(cand_df))
display(pred_df.head())
display(cand_df.head())

## Recommended AAAI Ablations

Run the notebook with these settings:

1. `RUN_DATASET = "iuxray_official"`, `DECODING_MODE = "standard"`
2. `RUN_DATASET = "iuxray_official"`, `DECODING_MODE = "graph_constrained"`
3. `RUN_DATASET = "mimic_aug"`, `DECODING_MODE = "standard"`
4. `RUN_DATASET = "mimic_aug"`, `DECODING_MODE = "graph_constrained"`

For final paper runs, set `SMOKE_RUN = False` and run the full train/test splits.